# Chatbot RAG des Gacetas — notebook unique (F)

**Tout le projet de F dans un seul notebook**, à exécuter de haut en bas : *Exécution → Tout exécuter*.

| Partie | Contenu | Durée indicative (GPU T4) |
|---|---|---|
| **A · J1** | PDF → chunks → TF-IDF → embeddings BERT | 5 à 15 min |
| **B · J2** | retriever dense FAISS, seuil τ | 1 min |
| **C · J3** | génération, refus, évaluation | 5 à 10 min |
| **D · J4** | gel des paramètres, doc d'ingestion | 1 min |
| **E · Interface** | lance Streamlit et affiche le lien du chatbot | 2 min |

**Pourquoi un seul notebook ?** Une seule session Colab, un seul GPU, aucune manipulation entre les étapes. Chaque partie **enregistre quand même ses fichiers sur le Drive** (`ProjectRag/data`, `config`, `eval`, `docs`, `app`) : si Colab se déconnecte, tu pourras relancer à partir de la partie où tu t'étais arrêté.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → GPU T4*.

In [ ]:
# Installation de tout ce qu'il faut pour les 5 parties (une seule fois par session)
!pip install -q pymupdf sentence-transformers nltk rank_bm25 faiss-cpu transformers accelerate pyyaml streamlit

---
# PARTIE A · J1 — Le socle de données

## 2. Connexion au Drive et chemins

Tes 49 PDF sont dans **Mon Drive → ProjectRag → PDF**, donc accessibles dans Colab à `/content/drive/MyDrive/ProjectRag/PDF`.

B et J doivent avoir le dossier au **même chemin** (s'il est partagé avec eux : raccourci dans *Mon Drive*, avec le nom exact `ProjectRag`), pour que le code marche chez tout le monde sans modification.

Les fichiers produits seront écrits dans `ProjectRag/data/` : sur le Drive, donc **permanents et partagés**. (Le disque de Colab, lui, s'efface à la fin de la session.)

In [ ]:
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/ProjectRag')
except ImportError:                      # exécution hors Colab (PC local)
    PROJECT = Path(os.environ.get('PROJECT_DIR', './ProjectRag'))

RAW_DIR   = PROJECT / 'PDF'               # entrée : on ne modifie jamais ces fichiers
DATA_DIR  = PROJECT / 'data'              # sortie : ce que ce notebook produit
TFIDF_DIR = DATA_DIR / 'tfidf'
BERT_DIR  = DATA_DIR / 'bert'
FIG_DIR   = DATA_DIR / 'figures'
for d in (DATA_DIR, TFIDF_DIR, BERT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

pdf_paths = sorted(RAW_DIR.glob('*.pdf'))
print(f'Dossier PDF : {RAW_DIR}')
print(f'{len(pdf_paths)} PDF trouvés')
assert pdf_paths, "Aucun PDF trouvé : vérifie le chemin MyDrive/ProjectRag/PDF (majuscules comprises)."

### Paramètres du J1

Tous les réglages sont regroupés ici. Au J4, ce sont eux que tu « figeras » dans `config.yaml`.

- `MAX_PDF` : mets un petit nombre (ex. 5) pour un premier essai rapide, puis `None` pour tout traiter.
- `CHUNK_SIZE` / `CHUNK_OVERLAP` : taille d'un passage et chevauchement entre deux passages voisins, **en caractères** (1 500 caractères ≈ 300 à 350 mots espagnols).
- `MIN_CHARS_PAGE` : sous ce nombre de caractères, une page est probablement une image scannée.

In [ ]:
MAX_PDF          = None      # ex. 5 pour tester vite ; None = tous les PDF
CHUNK_SIZE       = 1500      # caractères par chunk
CHUNK_OVERLAP    = 200       # caractères partagés entre deux chunks voisins
MIN_CHUNK_CHARS  = 200       # on jette les chunks plus petits (bruit)
MIN_CHARS_PAGE   = 100       # page « vide » en dessous (scan probable)
EMBEDDING_MODEL  = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SEED             = 42

if MAX_PDF:
    pdf_paths = pdf_paths[:MAX_PDF]
print(f'{len(pdf_paths)} PDF seront traités')

In [ ]:
import re, json, unicodedata, random
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3})
try:
    from IPython.display import display
except ImportError:
    display = print

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')
    plt.show()

## 3. Parsing : lire le texte des PDF

**Pourquoi :** un PDF n'est pas du texte, c'est une mise en page. Il faut en extraire le texte, page par page, en gardant **le numéro de page** : c'est lui qui permettra au chatbot de citer « Gaceta 1382, p. 5 ».

**Le piège à vérifier en premier :** si une Gaceta est un **scan** (une photo de papier), `fitz` renvoie une page vide. Dans ce cas il faut de l'OCR (Tesseract en espagnol), ce qui change ton planning. Le graphique ci-dessous te le dit tout de suite.

**Métadonnées :** le numéro de Gaceta vient du nom de fichier (`gaceta_1382_ded330bb6f.pdf` → `1382`). La date est cherchée dans le texte de la première page (forme « 12 de mayo de 2023 »).

In [ ]:
import fitz  # PyMuPDF

MESES = {'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,'julio':7,
         'agosto':8,'septiembre':9,'setiembre':9,'octubre':10,'noviembre':11,'diciembre':12}
DATE_RE = re.compile(r'(\d{1,2})\s+de\s+(' + '|'.join(MESES) + r')\s+de\s+(\d{4})', re.IGNORECASE)

def gaceta_id(path):
    m = re.search(r'gaceta_(\d+)', path.stem, re.IGNORECASE)
    return m.group(1) if m else path.stem

def find_date(text):
    m = DATE_RE.search(text)
    if not m:
        return None
    d, mes, y = m.groups()
    return f'{int(y):04d}-{MESES[mes.lower()]:02d}-{int(d):02d}'

pages = []      # une ligne par page : gaceta, page, texte brut
for i, path in enumerate(pdf_paths, 1):
    try:
        doc = fitz.open(path)
    except Exception as e:
        print(f'  ! illisible : {path.name} ({e})')
        continue
    gid = gaceta_id(path)
    for pno, page in enumerate(doc, 1):
        pages.append({'gaceta': gid, 'file': path.name, 'page': pno, 'raw': page.get_text()})
    doc.close()
    print(f'[{i}/{len(pdf_paths)}] {path.name} : {pno} pages')

pages_df = pd.DataFrame(pages)
pages_df['n_chars'] = pages_df['raw'].str.len()
dates = (pages_df[pages_df.page <= 2].groupby('gaceta')['raw']
         .apply(lambda t: find_date(' '.join(t))))
pages_df['date'] = pages_df['gaceta'].map(dates)
print(f'\nTotal : {len(pages_df)} pages, {pages_df.n_chars.sum():,} caractères')

### Visualisation : y a-t-il du texte sur chaque page ?

- **À gauche** : nombre de pages par Gaceta. On voit si certains PDF sont énormes.
- **À droite** : nombre de caractères par page. Les barres rouges à gauche de la ligne pointillée = pages quasi vides.

**Comment réagir :** quelques pages vides (couvertures, pages blanches) sont normales. Si **une grande partie** des pages est vide, ce sont des scans : préviens l'équipe, il faudra ajouter une étape d'OCR.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
per_g = pages_df.groupby('gaceta').size().sort_index()
ax1.bar(per_g.index.astype(str), per_g.values, color='#534AB7')
ax1.set_title('Pages par Gaceta'); ax1.set_xlabel('Gaceta'); ax1.set_ylabel('pages')
ax1.tick_params(axis='x', rotation=90, labelsize=7)

bins = np.linspace(0, pages_df.n_chars.quantile(.99) + 1, 40)
n, b, patches = ax2.hist(pages_df.n_chars, bins=bins, color='#1D9E75')
for p, left in zip(patches, b[:-1]):
    if left < MIN_CHARS_PAGE:
        p.set_color('#E24B4A')
ax2.axvline(MIN_CHARS_PAGE, ls='--', color='gray')
ax2.set_title('Caractères par page'); ax2.set_xlabel('caractères'); ax2.set_ylabel('pages')
savefig('03_parsing')

empty = pages_df[pages_df.n_chars < MIN_CHARS_PAGE]
share = len(empty) / len(pages_df)
print(f'Pages quasi vides : {len(empty)} / {len(pages_df)} ({share:.1%})')
if share > .2:
    print('⚠ Plus de 20 % de pages vides : probablement des scans → OCR nécessaire.')
display(empty.groupby('file').size().sort_values(ascending=False).head(10).rename('pages vides'))

### Aperçu d'une page brute

Regarde ce texte : tu vas y voir les « déchets » de mise en page que le §4 va retirer (en-têtes répétés, numéros de page, mots coupés en fin de ligne avec un tiret).

In [ ]:
sample = pages_df[pages_df.n_chars > 500].sample(1, random_state=SEED).iloc[0]
print(f"Gaceta {sample.gaceta} · page {sample.page} · date {sample.date}\n" + '─' * 70)
print(sample.raw[:1500])

## 4. Nettoyage

**Pourquoi :** les en-têtes (« GACETA DEL CONGRESO », « Página 12 »…) se répètent sur **chaque page**. Si on les garde, ils polluent TF-IDF (des mots présents partout) et les embeddings, et ils prennent de la place dans chaque chunk.

**Ce que fait la fonction `clean_page` :**
1. **Lignes répétées** : on regarde seulement les 3 premières et 3 dernières lignes de chaque page (là où sont les en-têtes et pieds de page). Une ligne qui y revient sur plus de 30 % des pages d'une même Gaceta est supprimée. On ne touche jamais au milieu de la page, ni aux lignes « ARTÍCULO ».
2. **Numéros de page** seuls sur une ligne → supprimés.
3. **Mots coupés** : `legisla-\ntivo` → `legislativo`.
4. **Retours à la ligne** au milieu d'une phrase → espace ; on garde les vrais paragraphes.
5. **Espaces multiples** → un seul.

On **garde les accents** (*ley, artículo, número*) : BERT en a besoin. Ils seront traités à part pour TF-IDF.

In [ ]:
PAGE_NUM_RE = re.compile(r'^\s*(p[aá]gina\s*)?\d{1,4}\s*$', re.IGNORECASE)
ARTICLE_RE  = re.compile(r'\s*ART[IÍ]CULO', re.IGNORECASE)   # jamais supprimé

def norm_line(l):
    # version « comparable » d'une ligne : chiffres remplacés, minuscules
    return re.sub(r'\d+', '#', l.strip().lower())

EDGE = 3   # nb de lignes examinées en haut et en bas de chaque page

def edge_lines(text):
    ls = [l for l in text.splitlines() if l.strip()]
    return ls[:EDGE] + ls[-EDGE:]

def repeated_lines(df, min_share=0.3):
    rep = {}
    for gid, g in df.groupby('gaceta'):
        cnt = Counter()
        for t in g['raw']:
            cnt.update({norm_line(l) for l in edge_lines(t) if len(l.strip()) < 150})
        rep[gid] = {l for l, c in cnt.items() if len(g) >= 3 and c / len(g) >= min_share}
    return rep

REPEATED = repeated_lines(pages_df)

def clean_page(text, repeated):
    all_lines = text.splitlines()
    nonempty = [i for i, l in enumerate(all_lines) if l.strip()]
    edges = set(nonempty[:EDGE] + nonempty[-EDGE:])
    lines = []
    for idx, l in enumerate(all_lines):          # les lignes vides sont gardées : elles marquent les paragraphes
        if idx in edges and not ARTICLE_RE.match(l) and (norm_line(l) in repeated or PAGE_NUM_RE.match(l)):
            continue
        lines.append(l.rstrip())
    t = '\n'.join(lines)
    t = unicodedata.normalize('NFC', t)
    t = re.sub(r'(\w)-\n(\w)', r'\1\2', t)            # mots coupés
    t = re.sub(r'\n{2,}', '\n\n', t)                   # paragraphes
    t = re.sub(r'(?<!\n)\n(?!\n)', ' ', t)             # sauts de ligne internes
    t = re.sub(r'[ \t]+', ' ', t)
    return t.strip()

pages_df['clean'] = [clean_page(t, REPEATED.get(g, set())) for t, g in zip(pages_df.raw, pages_df.gaceta)]
pages_df['n_clean'] = pages_df['clean'].str.len()

ex = next(iter(v for v in REPEATED.values() if v), set())
print('Exemples de lignes détectées comme en-têtes / pieds de page :')
for l in list(ex)[:8]:
    print('  •', l)

### Visualisation : combien de texte le nettoyage retire-t-il ?

Chaque point est une page : en abscisse sa longueur avant nettoyage, en ordonnée après. Sur la diagonale = rien retiré.

**Comment lire :** la plupart des points doivent être **juste sous** la diagonale (quelques en-têtes retirés). Si des points tombent très bas, le nettoyage est trop agressif : regarde ces pages avant de continuer.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.scatter(pages_df.n_chars, pages_df.n_clean, s=6, alpha=.4, color='#534AB7')
m = pages_df.n_chars.max()
ax1.plot([0, m], [0, m], ls='--', color='gray')
ax1.set_xlabel('caractères avant'); ax1.set_ylabel('caractères après')
ax1.set_title('Avant / après nettoyage (1 point = 1 page)')

removed = (1 - pages_df.n_clean.sum() / max(pages_df.n_chars.sum(), 1))
ax2.bar(['avant', 'après'], [pages_df.n_chars.sum(), pages_df.n_clean.sum()],
        color=['#B4B2A9', '#1D9E75'])
ax2.set_title(f'Volume total de texte ({removed:.1%} retiré)')
savefig('04_nettoyage')

In [ ]:
print('AVANT\n' + '─' * 70); print(sample.raw[:700])
print('\nAPRÈS\n' + '─' * 70); print(pages_df.loc[sample.name, 'clean'][:700])

## 5. Chunking : découper en passages

**Pourquoi découper ?**
- Une Gaceta fait des dizaines de pages : trop long pour un LLM et impossible à citer précisément.
- Un chunk trop **petit** perd le contexte (une phrase seule ne veut rien dire).
- Un chunk trop **grand** mélange plusieurs sujets : le score de ressemblance devient flou.

**La méthode :** on découpe de préférence aux frontières naturelles, dans cet ordre : début d'un **ARTÍCULO**, paragraphe, phrase, puis mot. On remplit chaque chunk jusqu'à ~`CHUNK_SIZE` caractères.

**Le chevauchement (`overlap`) :** les 200 derniers caractères d'un chunk sont répétés au début du suivant. Ainsi une idée coupée en deux reste lisible dans au moins un des deux chunks.

**Un chunk par page ?** Non : on découpe **chaque page séparément**, pour que chaque chunk ait un numéro de page exact (indispensable pour les citations). Petit coût : un paragraphe à cheval sur deux pages est coupé.

In [ ]:
SEPARATORS = [r'\n\n(?=ART[IÍ]CULO)', r'\n\n', r'(?<=[.;:])\s+', r'\s+']

def split_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, seps=SEPARATORS):
    # 1) couper en morceaux qui tiennent dans `size`, en essayant les séparateurs dans l'ordre
    def atoms(t, level=0):
        if len(t) <= size or level >= len(seps):
            return [t] if len(t) <= size else [t[i:i + size] for i in range(0, len(t), size)]
        out = []
        for part in re.split(seps[level], t):
            if part.strip():
                out += atoms(part.strip(), level + 1) if len(part) > size else [part.strip()]
        return out
    # 2) regrouper les morceaux jusqu'à `size`, avec chevauchement
    chunks, cur = [], ''
    for a in atoms(text):
        if cur and len(cur) + 1 + len(a) > size:
            chunks.append(cur)
            tail = cur[-overlap:]
            tail = tail[tail.find(' ') + 1:] if ' ' in tail else tail   # ne pas couper un mot
            cur = (tail + ' ' + a).strip()
        else:
            cur = (cur + ' ' + a).strip() if cur else a
    if cur:
        chunks.append(cur)
    return chunks

chunks = []
for r in pages_df.itertuples():
    if r.n_clean < MIN_CHARS_PAGE:
        continue
    for k, txt in enumerate(split_text(r.clean)):
        if len(txt) < MIN_CHUNK_CHARS:
            continue
        chunks.append({'chunk_id': f'G{r.gaceta}_p{r.page:03d}_c{k:02d}',
                       'gaceta': r.gaceta, 'date': r.date, 'page': r.page,
                       'file': r.file, 'text': txt,
                       'n_chars': len(txt), 'n_words': len(txt.split())})

chunks_df = pd.DataFrame(chunks)
assert chunks_df.chunk_id.is_unique
print(f'{len(chunks_df)} chunks créés à partir de {pages_df.gaceta.nunique()} Gacetas')
display(chunks_df[['chunk_id', 'gaceta', 'page', 'n_words']].head())

### Visualisation : les chunks ont-ils une bonne taille ?

- **À gauche** : distribution de la longueur des chunks (en mots). Idéal : un gros pic vers 250-350 mots. Beaucoup de très petits chunks = fins de pages, normal mais à surveiller.
- **À droite** : nombre de chunks par Gaceta. Une Gaceta énorme produira beaucoup de chunks ; ce déséquilibre est normal, mais il faudra s'en souvenir lors de l'évaluation.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(chunks_df.n_words, bins=40, color='#534AB7')
med = chunks_df.n_words.median()
ax1.axvline(med, ls='--', color='#D85A30', label=f'médiane = {med:.0f} mots')
ax1.set_xlabel('mots par chunk'); ax1.set_ylabel('chunks'); ax1.legend()
ax1.set_title('Taille des chunks')

per_g = chunks_df.groupby('gaceta').size().sort_index()
ax2.bar(per_g.index.astype(str), per_g.values, color='#1D9E75')
ax2.set_title('Chunks par Gaceta'); ax2.tick_params(axis='x', rotation=90, labelsize=7)
savefig('05_chunking')
display(chunks_df[['n_chars', 'n_words']].describe().round(0))

### Visualisation : le chevauchement entre deux chunks voisins

Le passage <span style="color:#D85A30">**surligné**</span> apparaît à la fin du chunk *n* **et** au début du chunk *n+1*. C'est ça, l'overlap.

In [ ]:
from IPython.display import HTML
pairs = chunks_df.groupby(['gaceta', 'page']).filter(lambda g: len(g) >= 2)
if len(pairs):
    a, b = pairs.iloc[0], pairs.iloc[1]
    common = ''
    for L in range(min(len(a.text), len(b.text), CHUNK_OVERLAP + 50), 10, -1):
        if a.text.endswith(b.text[:L]):
            common = b.text[:L]; break
    hl = lambda s: f'<mark style="background:#FAC775">{s}</mark>'
    html = (f'<b>{a.chunk_id}</b> (fin) : …{a.text[-350:-len(common) or None]}{hl(common)}<br><br>'
            f'<b>{b.chunk_id}</b> (début) : {hl(common)}{b.text[len(common):len(common)+300]}…')
    display(HTML(html))
else:
    print('Pas de page avec 2 chunks dans cet échantillon.')

### Sauvegarde du contrat n°1 : `chunks.jsonl`

Une ligne JSON par chunk. **C'est le fichier que B et J attendent.** Dès qu'il est écrit, préviens-les.

In [ ]:
CHUNKS_PATH = DATA_DIR / 'chunks.jsonl'
with open(CHUNKS_PATH, 'w', encoding='utf-8') as f:
    for rec in chunks_df.to_dict('records'):
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f'✔ {CHUNKS_PATH}  ({CHUNKS_PATH.stat().st_size / 1e6:.1f} Mo, {len(chunks_df)} lignes)')
print('\nExemple de ligne :')
print(json.dumps({k: (v[:120] + '…' if k == 'text' else v)
                  for k, v in chunks_df.iloc[0].to_dict().items()}, ensure_ascii=False, indent=1))

## 6. Index TF-IDF — la baseline V0

**L'idée :** chaque chunk devient une ligne de nombres, une colonne par mot du vocabulaire.

- **TF** (*term frequency*) : un mot qui apparaît souvent **dans ce chunk** compte plus.
- **IDF** (*inverse document frequency*) : un mot présent **dans presque tous les chunks** (*proyecto, ley, congreso*) ne distingue rien → poids faible. Un mot rare (*pensional, minería*) → poids fort.
- Score TF-IDF = TF × IDF.

Pour chercher, on transforme la **question** avec le même vectorizer, puis on calcule la **similarité cosinus** avec chaque chunk (0 = rien en commun, 1 = identique).

**Réglages choisis :**
- `stop_words` espagnols : on ignore *el, la, de, que…*
- `strip_accents='unicode'` : *artículo* et *articulo* deviennent le même mot (les PDF sont parfois mal encodés).
- `ngram_range=(1, 2)` : on garde aussi les paires de mots (*salario mínimo*, *seguridad social*).
- `min_df=2` : un mot vu dans un seul chunk est souvent une faute d'OCR → ignoré.
- `sublinear_tf=True` : 10 occurrences ne valent pas 10 fois plus qu'une.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp, joblib

try:
    import nltk
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOP_ES = stopwords.words('spanish')
except Exception:
    STOP_ES = ('de la que el en y a los del se las por un para con no una su al lo como más pero '
               'sus le ya o este sí porque esta entre cuando muy sin sobre también me hasta hay '
               'donde quien desde todo nos durante todos uno les ni contra otros ese eso ante '
               'ellos e esto mí antes algunos qué unos yo otro otras otra él tanto esa estos '
               'mucho quienes nada muchos cual poco ella estar estas algunas algo nosotros').split()
STOP_ES = sorted({unicodedata.normalize('NFKD', w).encode('ascii', 'ignore').decode() for w in STOP_ES})

vectorizer = TfidfVectorizer(stop_words=STOP_ES, strip_accents='unicode', lowercase=True,
                             ngram_range=(1, 2), min_df=2, max_df=0.8, sublinear_tf=True)
X_tfidf = vectorizer.fit_transform(chunks_df.text)

vocab = vectorizer.get_feature_names_out()
density = X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print(f'Matrice TF-IDF : {X_tfidf.shape[0]} chunks × {X_tfidf.shape[1]} mots/paires')
print(f'Cases non nulles : {density:.3%}  → matrice « creuse », d\'où le format .npz')

### Visualisation : les mots que TF-IDF trouve importants

- **À gauche** : mots les plus **rares** (IDF élevé), ceux qui distinguent le mieux un chunk. Il ne doit pas y avoir de déchets (numéros, morceaux de mots) ; sinon, il faut améliorer le nettoyage.
- **À droite** : mots les plus **communs** (IDF faible), présents partout, donc peu utiles pour retrouver un passage.

In [ ]:
idf = pd.Series(vectorizer.idf_, index=vocab)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
top_rare = idf[idf.index.str.len() > 4].sort_values(ascending=False).head(20)
top_comm = idf.sort_values().head(20)
ax1.barh(top_rare.index[::-1], top_rare.values[::-1], color='#534AB7'); ax1.set_title('IDF élevé : mots discriminants')
ax2.barh(top_comm.index[::-1], top_comm.values[::-1], color='#B4B2A9'); ax2.set_title('IDF faible : mots présents partout')
for ax in (ax1, ax2): ax.set_xlabel('IDF'); ax.tick_params(axis='y', labelsize=8)
savefig('06_tfidf_idf')

### Test : une première recherche avec TF-IDF

Voici, en miniature, ce que fera la fonction `retrieve()` de B : question → vecteur → similarité avec tous les chunks → les **k** meilleurs.

Change la question pour tester. Essaie aussi une question **hors corpus** (ex. sur le football) : les scores doivent être très bas. C'est ce constat qui justifiera le **seuil** du J2.

In [ ]:
def search_tfidf(question, k=5):
    q = vectorizer.transform([question])
    scores = cosine_similarity(q, X_tfidf).ravel()
    top = np.argsort(-scores)[:k]
    return chunks_df.iloc[top].assign(score=scores[top])

QUESTIONS = ['¿Qué dice el proyecto de ley sobre el salario mínimo?',
             '¿Quién ganó la Copa del Mundo de fútbol?']          # la 2e est hors corpus

fig, axes = plt.subplots(1, len(QUESTIONS), figsize=(13, 3.5), sharey=True)
for ax, qst in zip(axes, QUESTIONS):
    res = search_tfidf(qst)
    ax.barh(res.chunk_id[::-1], res.score[::-1], color='#1D9E75')
    ax.set_title(qst[:45] + '…', fontsize=9); ax.set_xlabel('score cosinus'); ax.set_xlim(0, 1)
savefig('06_tfidf_recherche')

res = search_tfidf(QUESTIONS[0], k=1).iloc[0]
print(f'Meilleur passage pour : « {QUESTIONS[0]} »\n→ {res.chunk_id} (score {res.score:.2f})\n{res.text[:500]}…')

### Sauvegarde : `tfidf_vectorizer.pkl` + `tfidf_matrix.npz`

- Le **vectorizer** contient le vocabulaire et les poids IDF : indispensable pour transformer les **futures questions** exactement comme les chunks.
- La **matrice** contient les chunks déjà transformés (ligne *i* = chunk *i* de `chunks.jsonl`).

In [ ]:
joblib.dump(vectorizer, TFIDF_DIR / 'tfidf_vectorizer.pkl')
sp.save_npz(TFIDF_DIR / 'tfidf_matrix.npz', X_tfidf.tocsr())
for p in sorted(TFIDF_DIR.iterdir()):
    print(f'✔ {p.name:25s} {p.stat().st_size / 1e6:6.1f} Mo')

## 6 bis. (Optionnel) Index BM25

L'équipe a un doute sur **qui construit BM25** (barré dans la case de B au J3, mais nécessaire pour l'hybride de J). Comme c'est un travail de données, très rapide, tu peux le construire ici avec **les mêmes chunks** et proposer de le livrer dès le J1. Passe `BUILD_BM25 = True` si l'équipe est d'accord.

In [ ]:
BUILD_BM25 = False

if BUILD_BM25:
    from rank_bm25 import BM25Okapi
    analyzer = vectorizer.build_analyzer()           # même tokenisation que TF-IDF
    tokenize = lambda t: [w for w in analyzer(t) if ' ' not in w]   # mots seuls
    bm25 = BM25Okapi([tokenize(t) for t in chunks_df.text])
    joblib.dump(bm25, DATA_DIR / 'bm25.pkl')
    print('✔ bm25.pkl')

## 7. BERT : des vecteurs de sens (préparation du J2)

**L'idée :** un modèle BERT lit chaque chunk et le résume en un vecteur de **384 nombres**. Deux textes qui **veulent dire la même chose** ont des vecteurs proches, même s'ils n'ont aucun mot en commun (*salario mínimo* ≈ *remuneración básica*). C'est ce que TF-IDF ne sait pas faire.

**Modèle choisi :** `paraphrase-multilingual-MiniLM-L12-v2`. Multilingue (gère l'espagnol), léger, rapide sur le GPU gratuit de Colab. Au J2, J testera des modèles plus gros (*multilingual-e5*, embeddings LLM) : on comparera.

**Astuce Colab :** *Exécution → Modifier le type d'exécution → GPU T4* accélère fortement cette étape.

**Normalisation :** on normalise chaque vecteur (longueur 1). La similarité cosinus devient alors un simple produit scalaire : c'est ce qu'utilisera FAISS au J2.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Calcul sur : {device}')
model = SentenceTransformer(EMBEDDING_MODEL, device=device)

embeddings = model.encode(chunks_df.text.tolist(), batch_size=64, show_progress_bar=True,
                          normalize_embeddings=True, convert_to_numpy=True).astype('float32')
print(f'Embeddings : {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions')
print(f'Taille en mémoire : {embeddings.nbytes / 1e6:.1f} Mo (matrice pleine)')

### Visualisation : la carte du sens

On ne peut pas dessiner 384 dimensions. La **PCA** les écrase en 2 dimensions en gardant le maximum d'information. Chaque point = un chunk, chaque couleur = une Gaceta.

**Comment lire :** des points proches parlent de sujets proches. Si les couleurs se regroupent, chaque Gaceta a ses propres thèmes. Si elles se mélangent, plusieurs Gacetas traitent des mêmes sujets : le retriever devra bien les départager.

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2, random_state=SEED).fit_transform(embeddings)

g_codes = chunks_df.gaceta.astype('category')
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=g_codes.cat.codes, cmap='tab20', s=8, alpha=.6)
ax.set_title('Chunks projetés en 2D (PCA des embeddings) · couleur = Gaceta')
ax.set_xticks([]); ax.set_yticks([])
if g_codes.cat.categories.size <= 20:
    handles = [plt.Line2D([], [], marker='o', ls='', color=sc.cmap(sc.norm(i)), label=c)
               for i, c in enumerate(g_codes.cat.categories)]
    ax.legend(handles=handles, title='Gaceta', fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
savefig('07_bert_carte')

### Comparaison TF-IDF vs BERT sur les mêmes questions

Pour chaque question, on compare les **5 chunks** trouvés par chaque méthode.

**Comment lire :**
- Les **scores ne sont pas comparables** d'une méthode à l'autre (BERT donne rarement moins de 0,1 ; TF-IDF descend à 0). On compare les **passages trouvés**, pas les chiffres.
- **Recouvrement** = combien de chunks sont trouvés par les deux méthodes. Faible recouvrement = elles voient des choses différentes ; c'est exactement l'argument de la recherche **hybride** (V3).
- Question hors corpus : regarde l'écart entre son meilleur score et celui d'une vraie question. Plus l'écart est grand, plus le **seuil** du J2 sera facile à régler.

In [ ]:
def search_bert(question, k=5):
    q = model.encode([question], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores = embeddings @ q[0]
    top = np.argsort(-scores)[:k]
    return chunks_df.iloc[top].assign(score=scores[top])

rows = []
fig, axes = plt.subplots(len(QUESTIONS), 2, figsize=(13, 3.2 * len(QUESTIONS)))
axes = np.atleast_2d(axes)
for i, qst in enumerate(QUESTIONS):
    rt, rb = search_tfidf(qst), search_bert(qst)
    for j, (res, name, col) in enumerate([(rt, 'TF-IDF', '#1D9E75'), (rb, 'BERT', '#534AB7')]):
        ax = axes[i, j]
        ax.barh(res.chunk_id[::-1], res.score[::-1], color=col)
        ax.set_xlim(0, 1); ax.set_title(f'{name} · {qst[:40]}…', fontsize=9)
    rows.append({'question': qst, 'top1 TF-IDF': round(rt.score.iloc[0], 3),
                 'top1 BERT': round(rb.score.iloc[0], 3),
                 'recouvrement top-5': len(set(rt.chunk_id) & set(rb.chunk_id))})
savefig('07_tfidf_vs_bert')
display(pd.DataFrame(rows))

### Sauvegarde : `embeddings.npy` + `embedding_config.json`

- `embeddings.npy` : la matrice (ligne *i* = chunk *i*).
- `embedding_config.json` : la « fiche d'identité ». Une question doit être encodée **avec exactement le même modèle**, sinon les scores n'ont aucun sens. B et J liront ce fichier pour savoir quel modèle charger.

In [ ]:
np.save(BERT_DIR / 'embeddings.npy', embeddings)
config = {'model': EMBEDDING_MODEL, 'dim': int(embeddings.shape[1]), 'n_chunks': int(embeddings.shape[0]),
          'normalized': True, 'similarity': 'dot product (= cosinus car normalisé)',
          'query_prefix': '', 'passage_prefix': '',
          'chunk_size': CHUNK_SIZE, 'chunk_overlap': CHUNK_OVERLAP,
          'chunks_file': 'data/chunks.jsonl'}
with open(BERT_DIR / 'embedding_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
for p in sorted(BERT_DIR.iterdir()):
    print(f'✔ {p.name:25s} {p.stat().st_size / 1e6:6.2f} Mo')

## 8. Vérification finale

On **recharge tous les fichiers depuis le Drive**, comme le feront B et J, et on vérifie la règle d'or : même nombre de lignes partout, dans le même ordre. Si une ligne affiche ✘, ne partage pas les fichiers avant d'avoir corrigé.

In [ ]:
chk_chunks = pd.read_json(DATA_DIR / 'chunks.jsonl', lines=True, dtype={'gaceta': str})
chk_vec    = joblib.load(TFIDF_DIR / 'tfidf_vectorizer.pkl')
chk_X      = sp.load_npz(TFIDF_DIR / 'tfidf_matrix.npz')
chk_E      = np.load(BERT_DIR / 'embeddings.npy')
chk_cfg    = json.load(open(BERT_DIR / 'embedding_config.json', encoding='utf-8'))

i = len(chk_chunks) // 2
checks = {
    'chunks.jsonl lisible, chunk_id uniques'        : chk_chunks.chunk_id.is_unique,
    'TF-IDF : autant de lignes que de chunks'        : chk_X.shape[0] == len(chk_chunks),
    'BERT : autant de lignes que de chunks'          : chk_E.shape[0] == len(chk_chunks),
    'config BERT cohérente'                          : chk_cfg['n_chunks'] == len(chk_chunks) and chk_cfg['dim'] == chk_E.shape[1],
    'TF-IDF aligné (ligne i recalculée = ligne i)'   : np.allclose(chk_vec.transform([chk_chunks.text[i]]).toarray(), chk_X[i].toarray()),
    'BERT aligné (vecteur i recalculé ≈ ligne i)'    : float(model.encode([chk_chunks.text[i]], normalize_embeddings=True)[0] @ chk_E[i]) > .99,
    'vecteurs BERT normalisés'                       : np.allclose(np.linalg.norm(chk_E, axis=1), 1, atol=1e-3),
}
for name, ok in checks.items():
    print(('✔ ' if ok else '✘ ') + name)

### Récapitulatif des livrables du J1

In [ ]:
files = [p for p in sorted(DATA_DIR.rglob('*')) if p.is_file() and 'figures' not in p.parts]
summary = pd.DataFrame({'fichier': [str(p.relative_to(PROJECT)) for p in files],
                        'taille (Mo)': [round(p.stat().st_size / 1e6, 2) for p in files]})
display(summary)
print(f'''
Corpus : {pages_df.gaceta.nunique()} Gacetas · {len(pages_df)} pages · {len(chunks_df)} chunks
Chunks : médiane {chunks_df.n_words.median():.0f} mots · taille {CHUNK_SIZE} car. · overlap {CHUNK_OVERLAP}
TF-IDF : {X_tfidf.shape[1]} termes · BERT : {EMBEDDING_MODEL.split('/')[-1]} ({embeddings.shape[1]} dim)
Figures : {FIG_DIR.relative_to(PROJECT)}/ (réutilisables dans le rapport)
''')

## Message à envoyer à B et J

> Les données V0 sont prêtes dans `ProjectRag/data/` :
> - `chunks.jsonl` : 1 ligne par chunk (`chunk_id`, `gaceta`, `date`, `page`, `text`…)
> - `tfidf/tfidf_vectorizer.pkl` + `tfidf/tfidf_matrix.npz` : pour la baseline V0 (B : `retrieve()`)
> - `bert/embeddings.npy` + `bert/embedding_config.json` : pour le dense du J2
>
> Règle : la ligne *i* de chaque matrice = le chunk *i* de `chunks.jsonl`. Ne pas trier ni filtrer `chunks.jsonl` sans recalculer les index.
>
> J : pour tes 35 questions, utilise les `chunk_id` comme réponses attendues (`gold_chunk_ids`).

## Et ensuite (J2)

- Construire l'index **FAISS** à partir de `embeddings.npy` (retriever dense, V1).
- Avec les 35 questions de J, comparer les scores top-1 des questions **dans** et **hors** corpus pour choisir le **seuil** (V2).

---
# PARTIES B, C, D · J2 → J4 — Recherche dense, refus, évaluation, gel

## Rechargement des fichiers de la partie A

Cette cellule recharge depuis le Drive ce que la partie A vient d'écrire. Si tu reprends le travail un autre jour, tu peux **commencer directement ici** sans refaire la partie A.

In [ ]:
import os, re, json, time, hashlib, random, platform
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, scipy.sparse as sp

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/ProjectRag')
except ImportError:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './ProjectRag'))

DATA_DIR   = PROJECT / 'data'
EVAL_DIR   = PROJECT / 'eval'
RES_DIR    = EVAL_DIR / 'results'
CONFIG_DIR = PROJECT / 'config'
DOCS_DIR   = PROJECT / 'docs'
FIG_DIR    = DATA_DIR / 'figures'
for d in (EVAL_DIR, RES_DIR, CONFIG_DIR, DOCS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3})
try:
    from IPython.display import display, Markdown
except ImportError:
    display = print; Markdown = str

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight'); plt.show()

# ---- fichiers du J1
chunks_df  = pd.read_json(DATA_DIR / 'chunks.jsonl', lines=True, dtype={'gaceta': str})
vectorizer = joblib.load(DATA_DIR / 'tfidf' / 'tfidf_vectorizer.pkl')
X_tfidf    = sp.load_npz(DATA_DIR / 'tfidf' / 'tfidf_matrix.npz')
embeddings = np.load(DATA_DIR / 'bert' / 'embeddings.npy').astype('float32')
emb_cfg    = json.load(open(DATA_DIR / 'bert' / 'embedding_config.json', encoding='utf-8'))

assert len(chunks_df) == X_tfidf.shape[0] == embeddings.shape[0], 'Fichiers du J1 désalignés : relance le notebook J1.'
print(f'{len(chunks_df)} chunks · {chunks_df.gaceta.nunique()} Gacetas · modèle {emb_cfg["model"]} ({embeddings.shape[1]} dim)')

---
# J2 · V1-V2 — Recherche dense et seuil

## 9. Le retriever dense (V1) avec FAISS

**L'idée :** au J1, tu as transformé chaque chunk en vecteur de sens. Il reste à transformer la **question** de la même façon, puis à trouver les vecteurs de chunks les plus proches.

**Pourquoi FAISS ?** Comparer une question à tous les chunks un par un marche, mais FAISS (bibliothèque de Meta) le fait de façon optimisée, et c'est le standard pour les RAG. On utilise `IndexFlatIP` : recherche **exacte** par produit scalaire. Comme les vecteurs sont normalisés, produit scalaire = similarité cosinus.

**Le contrat avec B :** même signature que la `retrieve()` du V0 → `retrieve_dense(question, k)` renvoie une liste de dictionnaires `{chunk_id, gaceta, page, text, score}`, triée par score décroissant. B peut donc remplacer TF-IDF par dense sans rien changer d'autre.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(emb_cfg['model'])
QUERY_PREFIX = emb_cfg.get('query_prefix', '')

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(DATA_DIR / 'bert' / 'dense.faiss'))
print(f'Index FAISS : {index.ntotal} vecteurs → data/bert/dense.faiss')

def _as_results(ids, scores):
    rows = chunks_df.iloc[ids]
    return [{'chunk_id': r.chunk_id, 'gaceta': r.gaceta, 'page': int(r.page),
             'text': r.text, 'score': float(s)} for r, s in zip(rows.itertuples(), scores)]

def retrieve_dense(question, k=5):
    q = encoder.encode([QUERY_PREFIX + question], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(q, k)
    return _as_results(ids[0], scores[0])

def retrieve_tfidf(question, k=5):                  # baseline V0 (même contrat)
    q = vectorizer.transform([question])
    s = (X_tfidf @ q.T).toarray().ravel()            # vecteurs TF-IDF déjà normalisés → cosinus
    ids = np.argsort(-s)[:k]
    return _as_results(ids, s[ids])

for r in retrieve_dense('¿Qué dice el proyecto de ley sobre el salario mínimo?', k=3):
    print(f"{r['score']:.3f}  {r['chunk_id']}  {r['text'][:90]}…")

## 10. Le jeu de questions et l'évaluation de la recherche

### 10.1 Charger les questions de J (ou créer un jeu provisoire)

Format attendu de `eval/eval_questions.jsonl` (contrat n°3) :
```json
{"qid": "q07", "question": "¿…?", "in_corpus": true, "gold_chunk_ids": ["G1380_p004_c01"], "gold_gacetas": ["1380"], "reference_answer": "…"}
```
- `gold_chunk_ids` : les chunks qui contiennent la réponse (le plus précis).
- `gold_gacetas` : si J ne connaît que la Gaceta, c'est accepté (une réponse est alors « trouvée » si un chunk de cette Gaceta remonte).
- `in_corpus: false` pour les questions **hors corpus** (dont la réponse n'est dans aucune Gaceta).

**Si le fichier n'existe pas**, on fabrique un jeu provisoire : des questions tirées de vrais chunks (réponse connue) + des questions hors sujet écrites à la main. ⚠ Ces questions reprennent les mots des chunks, ce qui **avantage TF-IDF**. Les vrais résultats seront ceux du jeu de J.

In [ ]:
EVAL_PATH = EVAL_DIR / 'eval_questions.jsonl'

OUT_OF_CORPUS = [
    '¿Quién ganó la Copa Mundial de fútbol de 2022?',
    '¿Cuál es la receta tradicional de la paella valenciana?',
    '¿Qué temperatura hace hoy en Tokio?',
    '¿Cómo se instala Python en un computador con Windows?',
    '¿Cuál es la capital de Australia?',
    '¿Cuánto cuesta un bitcoin actualmente?',
    '¿Quién dirigió la película Titanic?',
    '¿Cuántas lunas tiene el planeta Júpiter?',
]

def make_provisional_set(n_in=27, seed=SEED):
    rng = random.Random(seed)
    rows, pool = [], chunks_df[chunks_df.n_words > 80].sample(frac=1, random_state=seed)
    for r in pool.itertuples():
        sents = [s for s in re.split(r'(?<=[.;:])\s+', r.text) if 12 <= len(s.split()) <= 40]
        if not sents:
            continue
        words = rng.choice(sents).split()
        start = rng.randint(0, max(0, len(words) - 10))
        frag = ' '.join(words[start:start + 9]).strip(' ,.;:')
        rows.append({'qid': f'p{len(rows):02d}', 'question': f'¿Qué se dice sobre «{frag}»?',
                     'in_corpus': True, 'gold_chunk_ids': [r.chunk_id], 'gold_gacetas': [r.gaceta],
                     'reference_answer': ''})
        if len(rows) == n_in:
            break
    rows += [{'qid': f'o{i:02d}', 'question': q, 'in_corpus': False, 'gold_chunk_ids': [],
              'gold_gacetas': [], 'reference_answer': ''} for i, q in enumerate(OUT_OF_CORPUS)]
    return pd.DataFrame(rows)

if EVAL_PATH.exists():
    eval_df = pd.read_json(EVAL_PATH, lines=True, dtype={'gold_gacetas': object})
    EVAL_SOURCE = 'jeu de J'
else:
    eval_df = make_provisional_set()
    eval_df.to_json(EVAL_DIR / 'eval_questions_PROVISOIRE.jsonl', orient='records', lines=True, force_ascii=False)
    EVAL_SOURCE = 'PROVISOIRE (à remplacer par le jeu de J)'

for col in ('gold_chunk_ids', 'gold_gacetas'):
    if col not in eval_df: eval_df[col] = [[] for _ in range(len(eval_df))]
    eval_df[col] = eval_df[col].apply(lambda v: [str(x) for x in v] if isinstance(v, list) else [])
eval_df['in_corpus'] = eval_df['in_corpus'].astype(bool)

print(f'Jeu d\'évaluation : {EVAL_SOURCE}')
print(f'{eval_df.in_corpus.sum()} questions dans le corpus · {(~eval_df.in_corpus).sum()} hors corpus')
display(eval_df[['qid', 'question', 'in_corpus']].head(8))

### 10.2 Les métriques de recherche

On ne regarde que les questions **dans le corpus** (pour les hors corpus, il n'y a rien à trouver).

- **Recall@k** : proportion de questions pour lesquelles un bon chunk est dans les **k** premiers résultats. « Est-ce que la bonne réponse est au moins dans le panier ? »
- **MRR** (*Mean Reciprocal Rank*) : moyenne de 1/rang du premier bon chunk (rang 1 → 1, rang 2 → 0,5, rang 5 → 0,2, absent → 0). « Est-ce que la bonne réponse est **en haut** du panier ? »

Le LLM ne lit que les k passages qu'on lui donne : si le bon chunk n'y est pas, il ne **peut pas** bien répondre. La recherche est donc le plafond de tout le système.

In [ ]:
K_MAX = 10

def is_relevant(hit, row):
    if row.gold_chunk_ids:
        return hit['chunk_id'] in row.gold_chunk_ids
    return hit['gaceta'] in row.gold_gacetas

def first_relevant_rank(results, row):
    for rank, hit in enumerate(results, 1):
        if is_relevant(hit, row):
            return rank
    return None

def eval_retriever(fn, name, ks=(1, 3, 5, 10)):
    ins = eval_df[eval_df.in_corpus]
    ranks = [first_relevant_rank(fn(r.question, K_MAX), r) for r in ins.itertuples()]
    out = {'retriever': name}
    for k in ks:
        out[f'Recall@{k}'] = np.mean([rk is not None and rk <= k for rk in ranks])
    out['MRR'] = np.mean([1 / rk if rk else 0 for rk in ranks])
    return out, ranks

t0 = time.time()
res_v0, ranks_v0 = eval_retriever(retrieve_tfidf, 'V0 · TF-IDF')
res_v1, ranks_v1 = eval_retriever(retrieve_dense, 'V1 · Dense')
retrieval_table = pd.DataFrame([res_v0, res_v1]).set_index('retriever').round(3)
display(retrieval_table)
print(f'({time.time() - t0:.1f} s)')

### Visualisation : V0 contre V1

- **À gauche** : Recall@k pour k = 1, 3, 5, 10. Plus la courbe est haute, mieux c'est. L'écart entre les courbes = ce qu'apporte le dense.
- **À droite** : à quel rang le bon chunk apparaît-il ? Beaucoup de barres au rang 1 = excellent. La barre « absent » = questions ratées : ce sont elles qu'il faut aller lire pour comprendre les erreurs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ks = [1, 3, 5, 10]
for res, col in [(res_v0, '#1D9E75'), (res_v1, '#534AB7')]:
    ax1.plot(ks, [res[f'Recall@{k}'] for k in ks], marker='o', color=col, label=res['retriever'])
ax1.set_xticks(ks); ax1.set_ylim(0, 1.05); ax1.set_xlabel('k'); ax1.set_ylabel('Recall@k')
ax1.set_title('Le bon chunk est-il dans le top-k ?'); ax1.legend()

labels = ['1', '2', '3', '4-5', '6-10', 'absent']
def bucket(r):
    return 'absent' if r is None else '1' if r == 1 else '2' if r == 2 else '3' if r == 3 else '4-5' if r <= 5 else '6-10'
x = np.arange(len(labels)); w = .38
for off, ranks, col, name in [(-w/2, ranks_v0, '#1D9E75', 'V0'), (w/2, ranks_v1, '#534AB7', 'V1')]:
    c = Counter(bucket(r) for r in ranks)
    ax2.bar(x + off, [c[l] for l in labels], w, color=col, label=name)
ax2.set_xticks(x); ax2.set_xticklabels(labels); ax2.set_xlabel('rang du premier bon chunk')
ax2.set_ylabel('questions'); ax2.set_title('Où se trouve la bonne réponse ?'); ax2.legend()
savefig('10_recherche_v0_v1')

missed = eval_df[eval_df.in_corpus].assign(rank_v1=ranks_v1)
missed = missed[missed.rank_v1.isna()]
print(f'{len(missed)} question(s) ratée(s) par le dense dans le top-{K_MAX} :')
for r in missed.head(5).itertuples():
    print('  •', r.question)

## 11. Le seuil de score (V2) : savoir dire « je ne sais pas »

**Le problème :** `retrieve_dense` renvoie **toujours** k passages, même pour « ¿Quién ganó la Copa Mundial? ». Si on les donne au LLM, il risque d'inventer une réponse à partir de textes sans rapport.

**La solution :** si **même le meilleur passage** a un score inférieur à un seuil **τ**, on refuse **avant** d'appeler le LLM. C'est l'**étage 1** du refus.

**Comment choisir τ ?** On regarde le score top-1 de chaque question :
- les questions **dans le corpus** devraient avoir un score top-1 **élevé** ;
- les questions **hors corpus**, un score top-1 **bas**.

Le bon τ sépare les deux groupes. Deux erreurs possibles :
- **faux refus** : on refuse une vraie question (τ trop haut) → chatbot frustrant ;
- **refus manqué** : on laisse passer une question hors corpus (τ trop bas) → risque d'hallucination.

On choisit le τ qui maximise **(taux de bons refus − taux de faux refus)** (indice de Youden). Si plusieurs valeurs sont aussi bonnes, on prend celle **du milieu** : un τ collé au bord d'un groupe basculerait au moindre changement. Tu peux ensuite le décaler à la main selon ce que l'équipe juge le plus grave.

In [ ]:
eval_df['top1_dense'] = [retrieve_dense(q, 1)[0]['score'] for q in eval_df.question]
eval_df['top1_tfidf'] = [retrieve_tfidf(q, 1)[0]['score'] for q in eval_df.question]

def refusal_rates(scores, in_corpus, tau):
    refused = scores < tau
    good_ref = refused[~in_corpus].mean() if (~in_corpus).any() else np.nan   # hors corpus refusées
    false_ref = refused[in_corpus].mean() if in_corpus.any() else np.nan     # vraies questions refusées
    return good_ref, false_ref

s, ic = eval_df.top1_dense.values, eval_df.in_corpus.values
taus = np.round(np.arange(0, 1.0001, 0.01), 2)
curve = pd.DataFrame([(t, *refusal_rates(s, ic, t)) for t in taus], columns=['tau', 'bons_refus', 'faux_refus'])
curve['youden'] = curve.bons_refus - curve.faux_refus
best_taus = curve.loc[curve.youden >= curve.youden.max() - 1e-9, 'tau']
TAU = float(np.round(best_taus.median(), 2))      # milieu de la zone optimale : plus robuste qu'un bord
best = curve.set_index('tau').loc[TAU]
print(f'τ choisi = {TAU:.2f}  →  bons refus {best.bons_refus:.0%} · faux refus {best.faux_refus:.0%}')

### Visualisation : le choix de τ

- **À gauche** : distribution du score top-1, en **violet** les questions dans le corpus, en **corail** les hors corpus. La ligne pointillée = τ. Idéal : tout le corail à gauche de la ligne, tout le violet à droite. La zone où les couleurs se chevauchent = les cas difficiles.
- **À droite** : quand τ augmente, on refuse plus de questions hors corpus (bien) **mais aussi** plus de vraies questions (mal). Le point retenu est celui où l'écart entre les deux courbes est maximal.

**Cette figure est l'une des plus importantes de ton rapport** : elle justifie le chiffre τ.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(min(s.min(), 0), max(s.max(), 1), 30)
ax1.hist(s[ic], bins=bins, alpha=.75, color='#534AB7', label='dans le corpus')
ax1.hist(s[~ic], bins=bins, alpha=.75, color='#D85A30', label='hors corpus')
ax1.axvline(TAU, ls='--', color='black', label=f'τ = {TAU:.2f}')
ax1.set_xlabel('score top-1 (dense)'); ax1.set_ylabel('questions'); ax1.legend()
ax1.set_title('Les deux groupes de questions sont-ils séparables ?')

ax2.plot(curve.tau, curve.bons_refus, color='#1D9E75', label='bons refus (hors corpus refusées)')
ax2.plot(curve.tau, curve.faux_refus, color='#E24B4A', label='faux refus (vraies questions refusées)')
ax2.axvline(TAU, ls='--', color='black')
ax2.set_xlabel('seuil τ'); ax2.set_ylabel('taux'); ax2.set_ylim(-.02, 1.02); ax2.legend(fontsize=8)
ax2.set_title('Compromis selon τ')
savefig('11_seuil_tau')

### V2 : le retriever avec seuil

`retrieve_v2` = `retrieve_dense` + seuil. S'il renvoie une **liste vide**, la génération refusera sans appeler le LLM. On compare V0, V1, V2 sur un tableau unique.

Pour V0, on calcule aussi un seuil optimal sur les scores TF-IDF, pour une comparaison équitable du refus.

In [ ]:
def retrieve_v2(question, k=5, tau=None):
    tau = TAU if tau is None else tau
    res = retrieve_dense(question, k)
    return res if res and res[0]['score'] >= tau else []

s_t = eval_df.top1_tfidf.values
tau_tfidf = max(taus, key=lambda t: np.nan_to_num(np.subtract(*refusal_rates(s_t, ic, t))))
rows = []
for name, scores, tau, res in [('V0 · TF-IDF (+seuil)', s_t, tau_tfidf, res_v0),
                               ('V1 · Dense (sans seuil)', s, -np.inf, res_v1),
                               ('V2 · Dense + seuil', s, TAU, res_v1)]:
    g, f = refusal_rates(scores, ic, tau)
    ok = eval_df.in_corpus & (scores >= tau)
    rows.append({'version': name, 'τ': None if tau == -np.inf else round(float(tau), 2),
                 'Recall@5': res['Recall@5'], 'MRR': res['MRR'],
                 'bons refus': g, 'faux refus': f})
v2_table = pd.DataFrame(rows).set_index('version').round(3)
display(v2_table)
v2_table.to_csv(RES_DIR / 'J2_recherche_V0_V1_V2.csv')
print('✔ eval/results/J2_recherche_V0_V1_V2.csv')

**Comment interpréter :**
- Le **seuil ne change pas Recall@5 ni MRR** : il ne modifie pas l'ordre des résultats, il décide seulement s'il faut répondre.
- **V1 sans seuil a 0 % de bons refus** : il répond à tout, c'est exactement le problème qu'on résout.
- Si les faux refus de V2 sont élevés, regarde les vraies questions sous τ : souvent elles sont mal formulées ou leur réponse est mal découpée dans les chunks.

**À envoyer à B et J :** la valeur de τ et la fonction `retrieve_v2` (même contrat que `retrieve()`).

---
# J3 · V3 — Génération, refus hors corpus, évaluation

## 12. La génération avec citations

**Ce que fait cette étape :** on donne au LLM la question + les k passages retenus, avec des **règles strictes** :
1. répondre **uniquement** à partir des passages ;
2. **citer** chaque information avec l'identifiant du chunk, ex. `[G1380_p004_c01]` ;
3. si les passages ne suffisent pas, répondre **exactement** la phrase de refus. C'est l'**étage 2** du refus : il attrape les questions qui ont passé le seuil mais dont les passages ne contiennent pas la réponse.

Le prompt est en espagnol, comme les Gacetas et les questions.

**Le LLM par défaut :** `Qwen2.5-1.5B-Instruct`, petit modèle open source qui parle espagnol et tient dans le GPU gratuit de Colab. Il est modeste : les résultats serviront surtout à **tester la méthode d'évaluation**.

**Brancher la chaîne de B :** remplace le contenu de `generate_answer(question, passages)` par l'appel à sa chaîne. Tant que la fonction renvoie un texte, toute la suite (§13-14) marche sans modification.

In [ ]:
REFUSAL = 'No encuentro esta información en las Gacetas.'
K_GEN = 5
LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_NEW_TOKENS = 256

SYSTEM_PROMPT = f'''Eres un asistente que responde preguntas sobre las Gacetas del Congreso de Colombia.
Reglas:
1. Responde ÚNICAMENTE con la información de los pasajes proporcionados.
2. Después de cada afirmación, cita el identificador del pasaje entre corchetes, por ejemplo [G1380_p004_c01].
3. Si los pasajes no permiten responder, responde exactamente: "{REFUSAL}"
4. Responde en español, de forma breve (máximo 5 frases).'''

def build_user_prompt(question, passages):
    ctx = '\n\n'.join(f"[{p['chunk_id']}] (Gaceta {p['gaceta']}, p. {p['page']})\n{p['text']}" for p in passages)
    return f'Pasajes:\n{ctx}\n\nPregunta: {question}'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                                           device_map='auto')

def generate_answer(question, passages):
    # ▼▼ Pour utiliser la chaîne LangChain de B : remplace ce bloc par  return chaine_B.invoke(...)
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_prompt(question, passages)}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(llm.device)
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    # ▲▲

def answer(question, k=K_GEN, tau=None):
    passages = retrieve_v2(question, k, tau)
    if not passages:                                         # étage 1 : seuil
        return {'answer': REFUSAL, 'refused_by': 'seuil', 'passages': []}
    text = generate_answer(question, passages)               # étage 2 : LLM
    return {'answer': text, 'refused_by': 'llm' if is_refusal(text) else None, 'passages': passages}

def is_refusal(text):
    t = text.lower()
    return 'no encuentro esta información' in t or 'no encuentro esa información' in t

demo = answer('¿Qué dice el proyecto de ley sobre el salario mínimo?')
print(demo['answer'])
print('\nPassages utilisés :', [p['chunk_id'] for p in demo['passages']])

## 13. Exécuter le chatbot sur toutes les questions

Chaque question passe par le pipeline complet (seuil → LLM). On garde tout : réponse, étage qui a refusé, passages, citations. Compte quelques minutes sur GPU.

In [ ]:
CIT_RE = re.compile(r'\[(G[^\]\s]+?_p\d+_c\d+)\]')

records = []
t0 = time.time()
for i, r in enumerate(eval_df.itertuples(), 1):
    out = answer(r.question)
    retrieved = [p['chunk_id'] for p in out['passages']]
    cited = sorted(set(CIT_RE.findall(out['answer'])))
    records.append({'qid': r.qid, 'question': r.question, 'in_corpus': r.in_corpus,
                    'answer': out['answer'], 'refused': out['refused_by'] is not None,
                    'refused_by': out['refused_by'], 'retrieved': retrieved, 'cited': cited,
                    'gold_hit_in_context': any(is_relevant(p, r) for p in out['passages'])})
    print(f'\r{i}/{len(eval_df)} questions', end='')
gen_df = pd.DataFrame(records)
print(f'  ({time.time() - t0:.0f} s)')
gen_df.to_json(RES_DIR / 'J3_generations.jsonl', orient='records', lines=True, force_ascii=False)
print('✔ eval/results/J3_generations.jsonl')

## 14. Évaluation de la génération

On mesure **4 choses**, chacune répond à une question précise :

| Critère | Question | Comment on le mesure |
|---|---|---|
| **Refus** | Refuse-t-il ce qu'il faut, et seulement ça ? | matrice de confusion dans / hors corpus × répond / refuse |
| **Citations valides** | Les `[chunk_id]` cités existent-ils dans les passages fournis ? | une citation inventée = hallucination de source |
| **Citations correctes** | Cite-t-il un **bon** chunk (celui de la réponse attendue) ? | intersection citations ∩ gold |
| **Ancrage** | Les phrases de la réponse viennent-elles des passages ? | % de mots pleins de chaque phrase présents dans les passages |

L'**ancrage** est un indicateur automatique simple, pas une preuve : une phrase peut réutiliser les mots des passages et dire autre chose. C'est pour ça qu'on exporte aussi une **grille de relecture manuelle** (§14.3).

In [ ]:
STOP = set(vectorizer.get_stop_words() or [])

def content_words(t):
    import unicodedata
    t = unicodedata.normalize('NFKD', t.lower()).encode('ascii', 'ignore').decode()
    return [w for w in re.findall(r'[a-z0-9]{3,}', t) if w not in STOP]

def grounding(answer_text, passages_text):
    src = set(content_words(passages_text))
    sents = [s for s in re.split(r'(?<=[.!?])\s+', CIT_RE.sub('', answer_text)) if len(content_words(s)) >= 3]
    if not sents:
        return np.nan
    return float(np.mean([np.mean([w in src for w in content_words(s)]) for s in sents]))

text_by_id = dict(zip(chunks_df.chunk_id, chunks_df.text))
gaceta_by_id = dict(zip(chunks_df.chunk_id, chunks_df.gaceta))
gold_by_q = {r.qid: r for r in eval_df.itertuples()}

def score_row(g):
    row = gold_by_q[g.qid]
    answered = not g.refused
    valid = [c for c in g.cited if c in g.retrieved]
    correct = [c for c in g.cited if c in row.gold_chunk_ids
               or (not row.gold_chunk_ids and gaceta_by_id.get(c) in row.gold_gacetas)] if g.in_corpus else []
    return pd.Series({
        'a_cité': answered and len(g.cited) > 0,
        'citations_valides': (len(valid) / len(g.cited)) if g.cited else np.nan,
        'citation_correcte': (len(correct) > 0) if (answered and g.in_corpus) else np.nan,
        'ancrage': grounding(g.answer, ' '.join(text_by_id[c] for c in g.retrieved)) if answered else np.nan,
    })

gen_df = pd.concat([gen_df.drop(columns=[c for c in ['a_cité', 'citations_valides', 'citation_correcte', 'ancrage'] if c in gen_df]),
                    gen_df.apply(score_row, axis=1)], axis=1)

### 14.1 Le refus : matrice de confusion et étage responsable

- **À gauche** : la matrice. Les deux cases **vertes** (diagonale) sont les bons comportements : répondre aux vraies questions, refuser les hors corpus. Les cases **rouges** sont les erreurs.
- **À droite** : **qui** a refusé ? Le seuil (étage 1, rapide et sans coût) ou le LLM (étage 2) ? Si le LLM rattrape beaucoup de hors corpus, c'est que τ est trop bas. S'il refuse beaucoup de vraies questions, le prompt est trop strict ou les passages ne contiennent pas la réponse.

In [ ]:
cm = pd.crosstab(gen_df.in_corpus.map({True: 'dans le corpus', False: 'hors corpus'}),
                 gen_df.refused.map({False: 'répond', True: 'refuse'})).reindex(
                 index=['dans le corpus', 'hors corpus'], columns=['répond', 'refuse'], fill_value=0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
good = np.array([[1, 0], [0, 1]])
ax1.imshow(good, cmap=plt.matplotlib.colors.ListedColormap(['#F7C1C1', '#C0DD97']), vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax1.text(j, i, cm.values[i, j], ha='center', va='center', fontsize=22)
ax1.set_xticks([0, 1]); ax1.set_xticklabels(cm.columns); ax1.set_yticks([0, 1]); ax1.set_yticklabels(cm.index)
ax1.grid(False); ax1.set_title('Refus : vert = bon comportement')

by = gen_df[gen_df.refused].groupby(['in_corpus', 'refused_by']).size().unstack(fill_value=0)
by = by.reindex(index=[True, False], columns=['seuil', 'llm'], fill_value=0)
ax2.bar(['dans le corpus\n(faux refus)', 'hors corpus\n(bons refus)'], by['seuil'], color='#534AB7', label='étage 1 : seuil')
ax2.bar(['dans le corpus\n(faux refus)', 'hors corpus\n(bons refus)'], by['llm'], bottom=by['seuil'], color='#D85A30', label='étage 2 : LLM')
ax2.set_ylabel('questions refusées'); ax2.legend(); ax2.set_title('Quel étage a refusé ?')
savefig('14_refus')

### 14.2 Tableau de bord de la génération

Chaque barre est un pourcentage (plus haut = mieux). Les définitions sont dans le tableau du début de §14.

- **Bonnes réponses tentées** : parmi les vraies questions, % auxquelles il a répondu.
- **Bons refus** : parmi les hors corpus, % refusées.
- **Contexte contient la réponse** : % des vraies questions dont un bon chunk était parmi les passages donnés au LLM. C'est le **plafond** : la génération ne peut pas faire mieux que ça.

In [ ]:
ins, oos = gen_df[gen_df.in_corpus], gen_df[~gen_df.in_corpus]
ans = gen_df[~gen_df.refused]
gen_metrics = {
    'Bonnes réponses tentées': (~ins.refused).mean() if len(ins) else np.nan,
    'Bons refus (hors corpus)': oos.refused.mean() if len(oos) else np.nan,
    'Contexte contient la réponse': ins.gold_hit_in_context.mean() if len(ins) else np.nan,
    'Réponses avec citation': ans['a_cité'].mean() if len(ans) else np.nan,
    'Citations valides': ans.citations_valides.mean(),
    'Citation correcte': ins[~ins.refused].citation_correcte.mean() if len(ins) else np.nan,
    'Ancrage moyen': ans.ancrage.mean(),
}
gm = pd.Series(gen_metrics).astype(float)
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(gm.index[::-1], gm.values[::-1], color='#534AB7')
for b, v in zip(bars, gm.values[::-1]):
    ax.text((0 if np.isnan(v) else v) + .01, b.get_y() + b.get_height() / 2, 'n/a' if np.isnan(v) else f'{v:.0%}', va='center')
ax.set_xlim(0, 1.1); ax.set_title(f'Génération V3 · τ = {TAU:.2f} · k = {K_GEN} · {LLM_MODEL.split("/")[-1]}')
savefig('14_generation_dashboard')
pd.DataFrame([gen_metrics]).round(3).to_csv(RES_DIR / 'J3_generation_metriques.csv', index=False)
print('✔ eval/results/J3_generation_metriques.csv')

### 14.3 Grille de relecture manuelle et exemples

Les métriques automatiques ne suffisent pas : un humain doit lire. On exporte une grille `J3_relecture_manuelle.csv` à remplir (0 ou 1) avec les colonnes **fidèle**, **pertinente**, **citation_ok**. Ouvre-la dans Google Sheets, partage-la avec J. Même 15 questions relues donnent un chiffre défendable dans le rapport.

Ci-dessous, les cas les plus intéressants : erreurs de refus et réponses peu ancrées.

In [ ]:
grid = gen_df[['qid', 'question', 'in_corpus', 'answer', 'cited', 'refused']].copy()
grid['passages'] = gen_df.retrieved.apply(lambda ids: '\n---\n'.join(f'[{c}] {text_by_id[c][:400]}' for c in ids))
for c in ['fidèle (0/1)', 'pertinente (0/1)', 'citation_ok (0/1)', 'commentaire']:
    grid[c] = ''
grid.to_csv(RES_DIR / 'J3_relecture_manuelle.csv', index=False)
print('✔ eval/results/J3_relecture_manuelle.csv\n')

def show(title, df, n=3):
    print(f'══ {title} ({len(df)}) ══')
    for g in df.head(n).itertuples():
        print(f'Q : {g.question}\nR : {g.answer[:400]}\n   cités : {g.cited}\n')

show('Faux refus (vraie question refusée)', gen_df[gen_df.in_corpus & gen_df.refused])
show('Refus manqués (hors corpus non refusée)', gen_df[~gen_df.in_corpus & ~gen_df.refused])
show('Réponses les moins ancrées', gen_df[~gen_df.refused].sort_values('ancrage').dropna(subset=['ancrage']))

### 14.4 Sensibilité : et si on changeait τ ?

On relance **seulement la décision de refus** pour plusieurs valeurs de τ (sans régénérer les réponses déjà produites : on applique le seuil aux scores top-1 déjà calculés). Ça montre à l'équipe ce que coûte chaque choix. C'est aussi une petite **ablation** du seuil : τ = 0 revient à « pas de seuil ».

In [ ]:
llm_refused = dict(zip(gen_df.qid, gen_df.refused_by == 'llm'))
rows = []
for t in sorted({0.0, round(TAU - .1, 2), TAU, round(TAU + .1, 2)}):
    seuil_ref = eval_df.top1_dense < t
    refused = seuil_ref | eval_df.qid.map(llm_refused).fillna(False)  # approx. : le LLM refuserait pareil
    rows.append({'τ': t, 'bons refus': refused[~eval_df.in_corpus].mean(),
                 'faux refus': refused[eval_df.in_corpus].mean(),
                 'appels LLM évités': seuil_ref.mean()})
sens = pd.DataFrame(rows).set_index('τ').round(3)
display(sens)

---
# J4 · V4 — Gel des paramètres et documentation d'ingestion

## 15. Geler les paramètres : `config.yaml`

**Pourquoi « geler » ?** Pendant 3 jours, on a testé des réglages. Pour le rendu, **tout le monde doit utiliser exactement les mêmes**, et quelqu'un d'autre doit pouvoir **refaire les mêmes résultats** dans un mois. On écrit donc :

| Fichier | Contenu |
|---|---|
| `config/config.yaml` | **tous** les paramètres choisis (chunking, modèles, k, τ, prompt…) |
| `config/requirements_freeze.txt` | les versions exactes des bibliothèques |
| `config/MANIFEST.json` | une empreinte (SHA-256) de chaque fichier de données : si un fichier change, on le voit |

**Règle à partir de maintenant :** plus personne ne met de valeur « en dur » dans son code. On lit `config.yaml`.

⚠ Les valeurs de chunking viennent de `embedding_config.json` (écrit au J1). Si tu as modifié le chunking depuis, relance d'abord le notebook J1.

In [ ]:
import yaml, sys, importlib.metadata as md_

CONFIG = {
    'version': 'V4',
    'frozen_at': time.strftime('%Y-%m-%d %H:%M'),
    'data': {'source_dir': 'PDF/', 'n_gacetas': int(chunks_df.gaceta.nunique()), 'n_chunks': int(len(chunks_df)),
             'chunks_file': 'data/chunks.jsonl'},
    'chunking': {'chunk_size_chars': emb_cfg.get('chunk_size'), 'chunk_overlap_chars': emb_cfg.get('chunk_overlap'),
                 'split_by_page': True, 'separators': ['ARTÍCULO', 'paragraphe', 'phrase', 'mot']},
    'tfidf': {k: (list(v) if isinstance(v, tuple) else v) for k, v in vectorizer.get_params().items()
              if k in ('ngram_range', 'min_df', 'max_df', 'sublinear_tf', 'strip_accents', 'lowercase')},
    'embeddings': {'model': emb_cfg['model'], 'dim': int(embeddings.shape[1]), 'normalized': True,
                   'query_prefix': QUERY_PREFIX, 'index': 'faiss.IndexFlatIP', 'index_file': 'data/bert/dense.faiss'},
    'retrieval': {'retriever': 'dense', 'k': K_GEN, 'tau': TAU, 'tau_method': 'Youden (bons refus - faux refus)',
                  'tau_eval_set': EVAL_SOURCE},
    'generation': {'llm': LLM_MODEL, 'max_new_tokens': MAX_NEW_TOKENS, 'do_sample': False,
                   'refusal_text': REFUSAL, 'system_prompt': SYSTEM_PROMPT,
                   'prompt_sha256': hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:16]},
    'seed': SEED,
}
with open(CONFIG_DIR / 'config.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False, width=120)

pkgs = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'pymupdf', 'sentence-transformers', 'faiss-cpu',
        'transformers', 'torch', 'nltk', 'pyyaml', 'rank-bm25', 'langchain']
lines = [f'# Python {platform.python_version()}']
for p in pkgs:
    try: lines.append(f'{p}=={md_.version(p)}')
    except md_.PackageNotFoundError: pass
(CONFIG_DIR / 'requirements_freeze.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')

def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(block), b''): h.update(b)
    return h.hexdigest()

manifest = {str(p.relative_to(PROJECT)): {'sha256': sha256(p), 'bytes': p.stat().st_size}
            for p in sorted(DATA_DIR.rglob('*')) if p.is_file() and 'figures' not in p.parts}
json.dump(manifest, open(CONFIG_DIR / 'MANIFEST.json', 'w'), indent=2)

print((CONFIG_DIR / 'config.yaml').read_text(encoding='utf-8')[:1500])
print(f'✔ config/config.yaml · requirements_freeze.txt · MANIFEST.json ({len(manifest)} fichiers)')

### 15.1 Test de reproductibilité

On **recharge** `config.yaml` et l'index FAISS **depuis le disque**, comme le ferait quelqu'un d'autre, et on vérifie que les décisions (réponse/refus) et les passages retrouvés sont **identiques** à ceux qu'on vient de mesurer. Si tout est ✔, le système est figé.

In [ ]:
cfg = yaml.safe_load(open(CONFIG_DIR / 'config.yaml', encoding='utf-8'))
idx2 = faiss.read_index(str(PROJECT / cfg['embeddings']['index_file']))
enc2 = SentenceTransformer(cfg['embeddings']['model'])

def retrieve_frozen(q):
    v = enc2.encode([cfg['embeddings']['query_prefix'] + q], normalize_embeddings=True).astype('float32')
    sc, ids = idx2.search(v, cfg['retrieval']['k'])
    return [] if sc[0][0] < cfg['retrieval']['tau'] else list(chunks_df.chunk_id.iloc[ids[0]])

same = [retrieve_frozen(r.question) == (g.retrieved) for r, g in zip(eval_df.itertuples(), gen_df.itertuples())]
mf = json.load(open(CONFIG_DIR / 'MANIFEST.json'))
checks = {
    'config.yaml relu sans erreur': isinstance(cfg, dict),
    'index FAISS relu = même taille': idx2.ntotal == index.ntotal,
    'mêmes passages et mêmes refus pour toutes les questions': all(same),
    'empreintes des données inchangées': all(sha256(PROJECT / p) == v['sha256'] for p, v in mf.items()),
}
for k_, ok in checks.items():
    print(('✔ ' if ok else '✘ ') + k_)

## 16. Documentation d'ingestion : `docs/INGESTION.md`

C'est le document qui explique **comment les données ont été préparées** : d'où elles viennent, chaque étape, chaque paramètre et pourquoi, les chiffres obtenus, comment tout reconstruire, et les limites connues. Il est **généré automatiquement** à partir des vrais chiffres, puis tu peux le compléter à la main (sections marquées *À compléter*).

Il sert à trois choses : le rapport final, l'oral, et permettre à n'importe qui de reconstruire l'index.

In [ ]:
def table_md(df):
    try: return df.to_markdown()
    except ImportError: return '```\n' + df.to_string() + '\n```'

def pct(x): return 'n/a' if x is None or (isinstance(x, float) and np.isnan(x)) else f'{x:.0%}'
nw = chunks_df.n_words
fig_rel = lambda n: f'../data/figures/{n}.png'

doc = f'''# Documentation d'ingestion — Chatbot RAG Gacetas

*Générée le {time.strftime("%Y-%m-%d")} · version {CONFIG["version"]} · configuration : `config/config.yaml`*

## 1. Sources

- **Corpus** : {CONFIG["data"]["n_gacetas"]} PDF de la *Gaceta del Congreso* de Colombie, dossier `PDF/` du Drive `ProjectRag`.
- Les PDF ne sont jamais modifiés. Tous les fichiers dérivés sont dans `data/`.
- *À compléter : période couverte, origine des fichiers (site du Congrès / Imprenta Nacional), présence de scans.*

## 2. Pipeline

| Étape | Outil | Paramètres | Sortie |
|---|---|---|---|
| Parsing | PyMuPDF (`page.get_text()`) | texte page par page, n° de Gaceta depuis le nom de fichier, date par regex | — |
| Nettoyage | regex | en-têtes/pieds répétés (3 lignes haut/bas, >30 % des pages), n° de page, césures, sauts de ligne | — |
| Chunking | découpe récursive par page | {CONFIG["chunking"]["chunk_size_chars"]} caractères, chevauchement {CONFIG["chunking"]["chunk_overlap_chars"]}, coupe sur ARTÍCULO > paragraphe > phrase > mot | `data/chunks.jsonl` |
| Index lexical (V0) | scikit-learn TF-IDF | {", ".join(f"{k}={v}" for k, v in CONFIG["tfidf"].items())}, mots vides espagnols | `data/tfidf/` |
| Index dense (V1) | {CONFIG["embeddings"]["model"]} + FAISS IndexFlatIP | {CONFIG["embeddings"]["dim"]} dimensions, vecteurs normalisés (cosinus) | `data/bert/` |
| Seuil (V2) | — | τ = {TAU:.2f} ({CONFIG["retrieval"]["tau_method"]}) | `config/config.yaml` |
| Génération (V3) | {LLM_MODEL} | k = {K_GEN}, sans échantillonnage, citations `[chunk_id]`, refus : « {REFUSAL} » | — |

## 3. Statistiques du corpus

- **{len(chunks_df)} chunks** · médiane **{nw.median():.0f} mots** (min {nw.min()}, max {nw.max()})
- Chunks par Gaceta : de {chunks_df.groupby("gaceta").size().min()} à {chunks_df.groupby("gaceta").size().max()}

![parsing]({fig_rel("03_parsing")})
![chunking]({fig_rel("05_chunking")})

## 4. Résultats

Jeu d'évaluation : **{EVAL_SOURCE}** ({int(eval_df.in_corpus.sum())} questions dans le corpus, {int((~eval_df.in_corpus).sum())} hors corpus).

### Recherche (J2)

{table_md(v2_table)}

![seuil]({fig_rel("11_seuil_tau")})

### Génération (J3)

| Métrique | Valeur |
|---|---|
''' + '\n'.join(f'| {k} | {pct(v)} |' for k, v in gen_metrics.items()) + f'''

![refus]({fig_rel("14_refus")})

## 5. Format des fichiers (contrats)

- `chunks.jsonl` : une ligne JSON par chunk : `chunk_id` (`G<gaceta>_p<page>_c<n>`), `gaceta`, `date`, `page`, `file`, `text`, `n_chars`, `n_words`.
- **Règle d'alignement** : la ligne *i* de `chunks.jsonl` = la ligne *i* de `tfidf_matrix.npz` = la ligne *i* de `embeddings.npy` = le vecteur *i* de `dense.faiss`.
- `retrieve(question, k)` → liste de `{{chunk_id, gaceta, page, text, score}}` triée par score décroissant ; liste vide = refus (seuil).

## 6. Reconstruire tout de zéro

1. Mettre les PDF dans `MyDrive/ProjectRag/PDF/`.
2. Exécuter le notebook **J1** (parsing → chunks → TF-IDF → embeddings).
3. Exécuter le notebook **J2-J4** (FAISS → seuil → génération → gel).
4. Vérifier les empreintes avec `config/MANIFEST.json` et les versions avec `config/requirements_freeze.txt`.

## 7. Limites connues

- Les chunks sont découpés **par page** : un paragraphe à cheval sur deux pages est coupé.
- La détection d'en-têtes est heuristique (lignes répétées) ; des en-têtes atypiques peuvent rester.
- Les pages scannées sans texte ne sont pas traitées (pas d'OCR).
- τ a été choisi sur le jeu « {EVAL_SOURCE} » : à recalibrer si le jeu d'évaluation change.
- L'« ancrage » est une mesure lexicale approximative, complétée par une relecture manuelle.
- *À compléter avec ce que vous observez.*
'''
(DOCS_DIR / 'INGESTION.md').write_text(doc, encoding='utf-8')
print(f'✔ docs/INGESTION.md ({len(doc.splitlines())} lignes)')
display(Markdown(doc[:3000]))

## Récapitulatif : ce que tu livres au J2, J3, J4

In [ ]:
groups = {'J2': [DATA_DIR / 'bert' / 'dense.faiss', RES_DIR / 'J2_recherche_V0_V1_V2.csv'],
          'J3': [RES_DIR / 'J3_generations.jsonl', RES_DIR / 'J3_generation_metriques.csv', RES_DIR / 'J3_relecture_manuelle.csv'],
          'J4': [CONFIG_DIR / 'config.yaml', CONFIG_DIR / 'requirements_freeze.txt', CONFIG_DIR / 'MANIFEST.json', DOCS_DIR / 'INGESTION.md']}
rows = [{'jour': j, 'fichier': str(p.relative_to(PROJECT)), 'présent': '✔' if p.exists() else '✘',
         'taille (ko)': round(p.stat().st_size / 1e3, 1) if p.exists() else None}
        for j, ps in groups.items() for p in ps]
display(pd.DataFrame(rows))
print(f'\nτ = {TAU:.2f} · k = {K_GEN} · jeu d\'évaluation : {EVAL_SOURCE}')
if 'PROVISOIRE' in EVAL_SOURCE:
    print('⚠ Relance tout le notebook quand J aura déposé eval/eval_questions.jsonl : τ et les métriques changeront.')

## Message à envoyer à l'équipe

> **J2** : `retrieve_v2(question, k)` = dense + seuil τ (valeur dans `config/config.yaml`). Même contrat que `retrieve()` ; une liste vide veut dire « refuser ». Index : `data/bert/dense.faiss`.
>
> **J3** : résultats de génération dans `eval/results/`. Grille de relecture manuelle à remplir : `J3_relecture_manuelle.csv`.
>
> **J4** : paramètres figés dans `config/config.yaml`. Plus de valeurs en dur dans le code. Doc d'ingestion : `docs/INGESTION.md`.
>
> **J** : pour le tableau V0 → V4, les chiffres V0/V1/V2 sont dans `J2_recherche_V0_V1_V2.csv`. **B** : pour l'ablation, τ = 0 correspond à « sans seuil » (voir §14.4).

---
# PARTIE E · Interface Streamlit

On libère d'abord la mémoire du GPU : l'application va charger son propre LLM.

**Comment l'application tourne « en même temps » que le notebook ?** La cellule de lancement démarre Streamlit **en arrière-plan** (`subprocess.Popen`). La cellule se termine tout de suite, mais le serveur continue de tourner tant que **cet onglet Colab reste ouvert**. Le tunnel Cloudflare donne un lien pour l'ouvrir dans ton navigateur.

In [ ]:
# Libérer le GPU avant de lancer l'application
import gc, torch
for name in ('llm', 'tok', 'model', 'encoder'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} Go")

## E.1 Écrire les fichiers de l'application sur le Drive

Deux fichiers dans `ProjectRag/app/` :
- `rag_core.py` : le **moteur** (chargement des index figés, `retrieve`, seuil, génération). Il lit tout dans `config/config.yaml` : aucune valeur en dur.
- `app.py` : l'**interface** Streamlit.

B et J pourront réutiliser `rag_core.py` directement dans leurs notebooks (`import rag_core`).

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/ProjectRag/app', exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/ProjectRag/app/rag_core.py
"""Moteur du chatbot Gacetas : chargement des fichiers figés + recherche + génération.

Aucune dépendance à Streamlit : ce module peut aussi être importé dans un notebook.
Tous les réglages viennent de config/config.yaml (gelé au J4).
"""
import json
import os
import re
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml

PROJECT = Path(os.environ.get("RAG_PROJECT", "/content/drive/MyDrive/ProjectRag"))
CIT_RE = re.compile(r"\[(G[^\]\s]+?_p\d+_c\d+)\]")

VERSIONS = {
    "V0 · TF-IDF": {"retriever": "tfidf", "threshold": False},
    "V1 · Dense": {"retriever": "dense", "threshold": False},
    "V2 · Dense + seuil": {"retriever": "dense", "threshold": True},
}


# ----------------------------------------------------------------- chargement
def load_config(project=PROJECT):
    with open(project / "config" / "config.yaml", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_index(project=PROJECT):
    """Charge chunks + TF-IDF + FAISS + encodeur. Long : à mettre en cache côté UI."""
    import faiss
    from sentence_transformers import SentenceTransformer

    cfg = load_config(project)
    chunks = pd.read_json(project / cfg["data"]["chunks_file"], lines=True, dtype={"gaceta": str})
    res = {
        "cfg": cfg,
        "chunks": chunks,
        "vectorizer": joblib.load(project / "data" / "tfidf" / "tfidf_vectorizer.pkl"),
        "X_tfidf": sp.load_npz(project / "data" / "tfidf" / "tfidf_matrix.npz"),
        "index": faiss.read_index(str(project / cfg["embeddings"]["index_file"])),
        "encoder": SentenceTransformer(cfg["embeddings"]["model"]),
    }
    n = len(chunks)
    assert res["X_tfidf"].shape[0] == n and res["index"].ntotal == n, \
        "Fichiers désalignés : relancer les notebooks J1 puis J2-J4."
    return res


def load_llm(model_name):
    """Charge le LLM de génération. Long et lourd : à mettre en cache côté UI."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    llm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
    return tok, llm


# ----------------------------------------------------------------- recherche
def _rows(res, ids, scores):
    df = res["chunks"].iloc[list(ids)]
    return [{"chunk_id": r.chunk_id, "gaceta": str(r.gaceta), "page": int(r.page),
             "date": getattr(r, "date", None), "text": r.text, "score": float(s)}
            for r, s in zip(df.itertuples(), scores)]


def retrieve(res, question, k=5, retriever="dense"):
    """Contrat commun : liste de {chunk_id, gaceta, page, text, score} triée par score."""
    if retriever == "tfidf":
        q = res["vectorizer"].transform([question])
        s = (res["X_tfidf"] @ q.T).toarray().ravel()
        ids = np.argsort(-s)[:k]
        return _rows(res, ids, s[ids])
    prefix = res["cfg"]["embeddings"].get("query_prefix", "")
    v = res["encoder"].encode([prefix + question], normalize_embeddings=True).astype("float32")
    scores, ids = res["index"].search(v, k)
    return _rows(res, ids[0], scores[0])


# ----------------------------------------------------------------- génération
def build_messages(cfg, question, passages):
    ctx = "\n\n".join(f"[{p['chunk_id']}] (Gaceta {p['gaceta']}, p. {p['page']})\n{p['text']}"
                      for p in passages)
    return [{"role": "system", "content": cfg["generation"]["system_prompt"]},
            {"role": "user", "content": f"Pasajes:\n{ctx}\n\nPregunta: {question}"}]


def generate(cfg, llm_bundle, question, passages):
    import torch

    tok, llm = llm_bundle
    msgs = build_messages(cfg, question, passages)
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=cfg["generation"]["max_new_tokens"], do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


def is_refusal(cfg, text):
    ref = cfg["generation"]["refusal_text"].lower().rstrip(".")
    return ref in text.lower() or "no encuentro esa información" in text.lower()


def answer(res, question, version="V2 · Dense + seuil", k=None, tau=None, llm_bundle=None):
    """Pipeline complet. Renvoie aussi la trace de chaque étape pour l'interface."""
    cfg = res["cfg"]
    k = k or cfg["retrieval"]["k"]
    tau = cfg["retrieval"]["tau"] if tau is None else tau
    v = VERSIONS[version]

    t0 = time.perf_counter()
    passages = retrieve(res, question, k, v["retriever"])
    t_ret = time.perf_counter() - t0
    top1 = passages[0]["score"] if passages else 0.0
    out = {"question": question, "version": version, "k": k, "tau": tau if v["threshold"] else None,
           "passages": passages, "top1": top1, "t_retrieve": t_ret, "t_generate": 0.0,
           "refused_by": None, "answer": "", "cited": []}

    if v["threshold"] and top1 < tau:                       # étage 1 : seuil
        out.update(refused_by="seuil", answer=cfg["generation"]["refusal_text"])
        return out
    if llm_bundle is None:                                  # mode recherche seule
        out["refused_by"] = None
        out["answer"] = None
        return out

    t1 = time.perf_counter()
    text = generate(cfg, llm_bundle, question, passages)    # étage 2 : LLM
    out["t_generate"] = time.perf_counter() - t1
    out["answer"] = text
    out["cited"] = sorted(set(CIT_RE.findall(text)))
    if is_refusal(cfg, text):
        out["refused_by"] = "llm"
    return out


# ----------------------------------------------------------------- évaluation
def load_eval(project=PROJECT):
    """Résultats des notebooks (s'ils existent) pour l'onglet Évaluation."""
    r = project / "eval" / "results"
    files = {"recherche": r / "J2_recherche_V0_V1_V2.csv", "generation": r / "J3_generation_metriques.csv"}
    out = {k: pd.read_csv(p) for k, p in files.items() if p.exists()}
    gen = r / "J3_generations.jsonl"
    if gen.exists():
        out["generations"] = pd.read_json(gen, lines=True)
    return out

In [ ]:
%%writefile /content/drive/MyDrive/ProjectRag/app/app.py
"""Interface Streamlit du chatbot RAG des Gacetas.

Lancer :  streamlit run app.py
Le dossier du projet se règle avec la variable RAG_PROJECT
(par défaut /content/drive/MyDrive/ProjectRag).
"""
import html
import re

import altair as alt
import pandas as pd
import streamlit as st

import rag_core as rc

st.set_page_config(page_title="Chatbot Gacetas", page_icon="⚖️", layout="wide")

EXAMPLES = [
    "¿Qué dice el proyecto de ley sobre el salario mínimo?",
    "¿Qué propone la reforma pensional?",
    "¿Quién ganó la Copa Mundial de fútbol de 2022?",   # hors corpus : doit être refusée
]
STOP = {"que", "qué", "el", "la", "los", "las", "de", "del", "en", "y", "a", "un", "una", "por",
        "para", "con", "se", "su", "sobre", "dice", "es", "cuál", "cuáles", "cómo", "quién"}


# ----------------------------------------------------------------- ressources (cache)
@st.cache_resource(show_spinner="Chargement des index (chunks, TF-IDF, FAISS, encodeur)…")
def get_index():
    return rc.load_index()


@st.cache_resource(show_spinner="Chargement du LLM (1 à 2 minutes la première fois)…")
def get_llm(name):
    return rc.load_llm(name)


try:
    RES = get_index()
except Exception as e:  # message clair si les notebooks n'ont pas tourné
    st.error(f"Impossible de charger le projet depuis `{rc.PROJECT}`.\n\n{e}\n\n"
             "Vérifie que les notebooks J1 et J2-J4 ont été exécutés.")
    st.stop()
CFG = RES["cfg"]

# ----------------------------------------------------------------- barre latérale
with st.sidebar:
    st.header("Réglages")
    version = st.radio("Version du retriever", list(rc.VERSIONS), index=2,
                       help="V0 = mots-clés · V1 = sens · V2 = sens + refus si score trop bas")
    k = st.slider("k : passages récupérés", 1, 10, int(CFG["retrieval"]["k"]))
    tau = st.slider("τ : seuil de refus", 0.0, 1.0, float(CFG["retrieval"]["tau"]), 0.01,
                    disabled=not rc.VERSIONS[version]["threshold"],
                    help=f"Valeur figée dans config.yaml : {CFG['retrieval']['tau']:.2f}")
    use_llm = st.toggle("Générer la réponse avec le LLM", value=True,
                        help="Désactivé : on affiche seulement les passages trouvés (instantané).")
    st.divider()
    st.caption("Corpus")
    c1, c2 = st.columns(2)
    c1.metric("Gacetas", CFG["data"]["n_gacetas"])
    c2.metric("Chunks", CFG["data"]["n_chunks"])
    st.caption(f"Encodeur : `{CFG['embeddings']['model'].split('/')[-1]}`  \n"
               f"LLM : `{CFG['generation']['llm'].split('/')[-1]}`  \nConfig : {CFG['version']} "
               f"({CFG.get('frozen_at', '')})")
    if st.button("Effacer la conversation", use_container_width=True):
        st.session_state.history = []
        st.rerun()

LLM = get_llm(CFG["generation"]["llm"]) if use_llm else None
st.session_state.setdefault("history", [])


# ----------------------------------------------------------------- rendu
def label(cid):
    m = re.match(r"G(.+?)_p(\d+)_c\d+", cid)
    return f"Gaceta {m.group(1)}, p. {int(m.group(2))}" if m else cid


def render_answer_text(text):
    return rc.CIT_RE.sub(lambda m: f" `📄 {label(m.group(1))}`", text)


def highlight(text, question, n=900):
    words = {w for w in re.findall(r"\w{4,}", question.lower()) if w not in STOP}
    t = html.escape(text[:n]) + ("…" if len(text) > n else "")
    for w in sorted(words, key=len, reverse=True):
        t = re.sub(rf"(?i)\b({re.escape(w)}\w*)", r"<mark>\1</mark>", t)
    return t


def scores_chart(out):
    df = pd.DataFrame([{"passage": f"{i}. {label(p['chunk_id'])}", "score": p["score"],
                        "cité": "cité dans la réponse" if p["chunk_id"] in out["cited"] else "non cité"}
                       for i, p in enumerate(out["passages"], 1)])
    bars = alt.Chart(df).mark_bar(cornerRadiusEnd=3).encode(
        x=alt.X("score:Q", scale=alt.Scale(domain=[0, 1]), title="score de similarité"),
        y=alt.Y("passage:N", sort=None, title=None),
        color=alt.Color("cité:N", scale=alt.Scale(domain=["cité dans la réponse", "non cité"],
                                                    range=["#534AB7", "#B4B2A9"]),
                        legend=alt.Legend(orient="bottom", title=None)),
        tooltip=["passage", alt.Tooltip("score:Q", format=".3f")])
    layers = [bars]
    if out["tau"] is not None:
        rule = alt.Chart(pd.DataFrame({"tau": [out["tau"]]})).mark_rule(
            strokeDash=[5, 4], color="#D85A30", size=2).encode(x="tau:Q")
        text = alt.Chart(pd.DataFrame({"tau": [out["tau"]], "y": [df.passage.iloc[0]]})).mark_text(
            align="left", dx=4, dy=-8, color="#D85A30").encode(x="tau:Q", y="y:N", text=alt.value("τ"))
        layers += [rule, text]
    return alt.layer(*layers).properties(height=38 * len(df) + 40)


def pipeline_dot(out):
    on, off = 'style="filled", fillcolor="#CECBF6", color="#534AB7"', 'color="#B4B2A9", fontcolor="#888780"'
    th = out["tau"] is not None
    ref1, ref2 = out["refused_by"] == "seuil", out["refused_by"] == "llm"
    gen = out["answer"] is not None and not ref1
    ok = gen and not ref2
    n = lambda cond: on if cond else off
    return f"""digraph {{ rankdir=LR; node [shape=box, style="rounded", fontname="sans-serif", fontsize=11];
      q [label="Question", {on}];
      r [label="retrieve · k={out['k']}\\ntop-1 = {out['top1']:.2f}", {on}];
      s [label="{'seuil τ = %.2f' % out['tau'] if th else 'pas de seuil'}", {n(th)}];
      g [label="LLM\\n{'%.1f s' % out['t_generate'] if gen else ''}", {n(gen)}];
      a [label="Réponse\\n+ citations", {n(ok)}];
      x1 [label="Refus\\nétage 1", {n(ref1)}];
      x2 [label="Refus\\nétage 2", {n(ref2)}];
      q -> r -> s; s -> g [label="≥ τ"]; s -> x1 [label="< τ"]; g -> a; g -> x2; }}"""


def render_turn(out):
    if out["refused_by"] == "seuil":
        st.warning(f"**{out['answer']}**  \nRefus étage 1 : le meilleur passage a un score de "
                   f"{out['top1']:.2f}, sous le seuil τ = {out['tau']:.2f}. Le LLM n'a pas été appelé.")
    elif out["answer"] is None:
        st.info("Mode recherche seule (LLM désactivé) : voici les passages trouvés.")
    elif out["refused_by"] == "llm":
        st.warning(f"**{out['answer']}**  \nRefus étage 2 : le seuil était franchi "
                   f"(top-1 = {out['top1']:.2f}) mais le LLM a jugé que les passages ne répondent pas.")
    else:
        st.markdown(render_answer_text(out["answer"]))
        bad = [c for c in out["cited"] if c not in {p["chunk_id"] for p in out["passages"]}]
        if bad:
            st.error(f"Citation(s) inventée(s) par le LLM (absentes des passages) : {', '.join(bad)}")
        elif out["answer"] and not out["cited"]:
            st.caption("⚠ Réponse sans citation.")

    m = st.columns(4)
    m[0].metric("Score top-1", f"{out['top1']:.2f}")
    m[1].metric("Passages", len(out["passages"]))
    m[2].metric("Recherche", f"{out['t_retrieve'] * 1000:.0f} ms")
    m[3].metric("Génération", f"{out['t_generate']:.1f} s" if out["t_generate"] else "—")

    t1, t2, t3 = st.tabs(["Scores des passages", "Sources", "Chemin dans le pipeline"])
    with t1:
        st.altair_chart(scores_chart(out), use_container_width=True)
    with t2:
        for i, p in enumerate(out["passages"], 1):
            star = " · cité" if p["chunk_id"] in out["cited"] else ""
            with st.expander(f"{i}. {label(p['chunk_id'])} · score {p['score']:.3f}{star}"):
                st.caption(f"`{p['chunk_id']}`" + (f" · {p['date']}" if p.get("date") else ""))
                st.markdown(highlight(p["text"], out["question"]), unsafe_allow_html=True)
    with t3:
        st.graphviz_chart(pipeline_dot(out))


# ----------------------------------------------------------------- pages
st.title("Chatbot des Gacetas del Congreso")
tab_chat, tab_eval, tab_about = st.tabs(["💬 Chat", "📊 Évaluation", "ℹ️ Comment ça marche"])

with tab_chat:
    if not st.session_state.history:
        st.caption("Pose une question en espagnol, ou essaie un exemple :")
        cols = st.columns(len(EXAMPLES))
        for c, ex in zip(cols, EXAMPLES):
            if c.button(ex, use_container_width=True):
                st.session_state.pending = ex
                st.rerun()

    for turn in st.session_state.history:
        with st.chat_message("user"):
            st.markdown(turn["question"])
        with st.chat_message("assistant"):
            st.caption(f"{turn['version']} · k = {turn['k']}" +
                       (f" · τ = {turn['tau']:.2f}" if turn["tau"] is not None else ""))
            render_turn(turn)

    question = st.chat_input("¿Qué quieres saber sobre las Gacetas?") or st.session_state.pop("pending", None)
    if question:
        with st.chat_message("user"):
            st.markdown(question)
        with st.chat_message("assistant"):
            with st.spinner("Recherche des passages et génération…"):
                out = rc.answer(RES, question, version, k, tau, LLM)
            st.session_state.history.append(out)
            st.rerun()

with tab_eval:
    ev = rc.load_eval()
    if not ev:
        st.info("Aucun résultat trouvé dans `eval/results/` : exécute d'abord le notebook J2-J4.")
    if "recherche" in ev:
        st.subheader("Recherche : V0 → V2")
        df = ev["recherche"].set_index(ev["recherche"].columns[0])
        st.dataframe(df.style.format(precision=3), use_container_width=True)
        long = df[["Recall@5", "MRR"]].reset_index().melt(id_vars=df.index.name or "index")
        st.altair_chart(alt.Chart(long).mark_bar().encode(
            x=alt.X("value:Q", title=None, scale=alt.Scale(domain=[0, 1])),
            y=alt.Y(f"{df.index.name or 'index'}:N", title=None),
            color=alt.Color("variable:N", scale=alt.Scale(range=["#534AB7", "#1D9E75"]), title=None),
            yOffset="variable:N"), use_container_width=True)
    if "generation" in ev:
        st.subheader("Génération et refus (V3)")
        g = ev["generation"].T.reset_index()
        g.columns = ["métrique", "valeur"]
        st.altair_chart(alt.Chart(g).mark_bar(color="#534AB7").encode(
            x=alt.X("valeur:Q", scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format="%")),
            y=alt.Y("métrique:N", sort=None, title=None),
            tooltip=[alt.Tooltip("valeur:Q", format=".0%")]), use_container_width=True)
    figs = [("11_seuil_tau", "Choix du seuil τ"), ("14_refus", "Matrice de refus")]
    shown = [(rc.PROJECT / "data" / "figures" / f"{f}.png", t) for f, t in figs]
    shown = [(p, t) for p, t in shown if p.exists()]
    if shown:
        st.subheader("Figures des notebooks")
        for col, (p, t) in zip(st.columns(len(shown)), shown):
            col.image(str(p), caption=t)
    if "generations" in ev:
        st.subheader("Toutes les réponses du jeu d'évaluation")
        gd = ev["generations"]
        st.dataframe(gd[[c for c in ["question", "in_corpus", "refused_by", "answer", "cited"] if c in gd]],
                     use_container_width=True, hide_index=True)

with tab_about:
    st.markdown(f"""
**Le chatbot répond uniquement à partir des Gacetas (le corpus) et cite ses sources.**

1. **Recherche** : la question est comparée aux {CFG['data']['n_chunks']} chunks.
   *V0* compare les mots (TF-IDF), *V1/V2* comparent le sens (embeddings + FAISS). On garde les **k** meilleurs.
2. **Seuil (V2)** : si le meilleur score est sous **τ**, la question est jugée hors corpus et refusée
   sans appeler le LLM (*refus étage 1*).
3. **Génération** : le LLM reçoit la question et les k passages, avec l'ordre de ne rien inventer et de citer
   chaque passage utilisé. S'il ne trouve pas la réponse, il refuse (*refus étage 2*).

**À essayer pendant la démo :** passe de V0 à V2 sur la même question et compare les passages ;
pose une question hors corpus en V1 (il répond quand même) puis en V2 (il refuse) ;
monte τ et regarde à partir de quand de vraies questions sont refusées.
""")
    st.graphviz_chart("""digraph { rankdir=LR; node [shape=box, style="rounded,filled", fillcolor="#EEEDFE",
      color="#534AB7", fontname="sans-serif"]; PDF -> Chunks -> "Index\\nTF-IDF / FAISS" -> retrieve -> "seuil τ" -> LLM -> "Réponse\\n+ citations"; }""")

## E.2 Lancer l'application et obtenir le lien

Streamlit tourne dans Colab sur le port 8501. Colab n'est pas accessible directement depuis Internet, donc on ouvre un **tunnel Cloudflare** gratuit, qui donne une adresse en `https://…trycloudflare.com`.

- Le premier chargement prend 1 à 2 minutes (index + LLM), ensuite c'est en cache.
- Le lien reste valable tant que cette session Colab est ouverte.
- Le lien est public : ne le diffuse qu'à ton équipe et aux évaluateurs.

In [ ]:
import subprocess, time, re, os

APP_DIR = '/content/drive/MyDrive/ProjectRag/app'
os.environ['RAG_PROJECT'] = '/content/drive/MyDrive/ProjectRag'

if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

!pkill -f "streamlit run" ; pkill -f cloudflared
app = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                       cwd=APP_DIR, env=dict(os.environ),
                       stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        url = m.group(0); break
print('👉 Ouvre ton chatbot ici :', url or 'pas encore de lien, relance cette cellule')

## E.3 En cas de problème

- **Page blanche ou erreur** : affiche le journal de Streamlit avec la cellule ci-dessous.
- **« Impossible de charger le projet »** : les notebooks J1 / J2-J4 n'ont pas tourné, ou le chemin `ProjectRag` est différent.
- **Tu modifies `app.py`** : relance la cellule `%%writefile`, puis clique sur *Rerun* dans la page web (pas besoin de relancer le tunnel).

In [ ]:
print(open('/content/streamlit.log').read()[-3000:])

In [ ]:
# Arrêter l'application
app.terminate(); tunnel.terminate(); print('arrêté')